# S6E9 — The `% 1000` Leak: a from-scratch 0.946 pipeline

Every top public notebook in this competition is a rank-average of other people's
submissions. This one trains from raw data and still lands in the same place, because
the synthetic generator leaks through the **low-order digits of `Annual_Income_USD`**.

**The headline number:** target-encoding `Annual_Income_USD % 1000` — the remainder
after dividing by 1000, which in genuine survey data is pure noise — scores
**AUC 0.618 on its own**. It is not noise. It is a fingerprint of how the data was made.

Everything below is measured, including the things that did not work.

| feature block (cumulative) | OOF AUC | delta |
|---|---|---|
| raw 13 columns | 0.941825 | — |
| **+ digits / moduli** | **0.943529** | **+0.00170** |
| + quantisation ladder | 0.943781 | +0.00025 |
| + value frequency | 0.944114 | +0.00033 |
| + ratios and interaction products | 0.943931 | **−0.00018** |
| + categorical crosses | 0.943945 | ±0.00001 (nothing) |
| everything *except* digits | 0.943282 | −0.00066 |
| **+ income target encoding** | **0.945746** | **+0.00163** |

Two blocks carry the whole thing: **digits** and **income target encoding**. The
ratio and interaction features that usually get bolted onto a Playground notebook
actively *hurt* here, and digits alone beat the entire kitchen sink with digits
removed.

> The target-encoding gain was re-measured with the encoder refit inside every
> outer fold, so it is not CV leakage. Loose encoding 0.945241, strict nested
> encoding 0.945281 — the gain survives the stricter protocol.

Smoothing matters more than the choice of keys. Extra keys made it worse (11 keys
0.945305, 15 keys 0.945193, both below 4 keys); turning the smoothing down made it
better, sharply from prior 200 to prior 10 and then flat:

| TE prior | OOF AUC |
|---|---|
| 200 | 0.945432 |
| 50 | 0.945621 |
| 10 and below | 0.94573 ± 0.00001 |

Heavy smoothing washes the leak out. The signal lives at fine value resolution,
which is exactly what you would expect from a generator artefact rather than from
a real income effect.

In [ ]:
import os, gc, glob, time, warnings, json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore")

SEED     = 42
N_SPLITS = 10
TARGET   = "Will_Buy_EV"
T0       = time.time()

def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

# Use the GPU when the session has one, but *prove* it works first. A visible
# nvidia-smi is not the same as a usable device - a small or already-occupied
# card will let CatBoost start and then die with "CUDA error 2: out of memory"
# halfway through your run. Cheaper to find out now.
def gpu_usable():
    try:
        import subprocess
        subprocess.check_output("nvidia-smi", stderr=subprocess.DEVNULL)
    except Exception:
        return False
    try:
        probe = np.random.rand(600, 4)
        lab = (probe[:, 0] > 0.5).astype(int)
        CatBoostClassifier(iterations=5, depth=3, task_type="GPU",
                           verbose=0, allow_writing_files=False).fit(probe, lab)
        xgb.XGBClassifier(n_estimators=5, max_depth=3, tree_method="hist",
                          device="cuda", verbosity=0).fit(probe, lab)
        return True
    except Exception as e:
        print(f"GPU present but unusable ({type(e).__name__}), falling back to CPU")
        return False

GPU = gpu_usable()
log(f"GPU in use: {GPU}")

## 1. Data

Paths are discovered by globbing so the notebook runs unchanged whether or not you
have the original dataset attached.

In [ ]:
def find(pattern, *fallbacks):
    for p in (pattern,) + fallbacks:
        hits = sorted(glob.glob(p, recursive=True))
        if hits:
            return hits[0]
    return None

COMP = "/kaggle/input/competitions/playground-series-s6e9"
train = pd.read_csv(find(f"{COMP}/train.csv", "/kaggle/input/**/playground-series-s6e9/train.csv",
                     "/kaggle/input/**/train.csv", "train.csv"))
test  = pd.read_csv(find(f"{COMP}/test.csv",  "/kaggle/input/**/playground-series-s6e9/test.csv",
                     "/kaggle/input/**/test.csv", "test.csv"))
log(f"train {train.shape} | test {test.shape}")

y      = (train.pop(TARGET) == "Yes").astype(int).to_numpy()
test_id = test.pop("id").to_numpy()
train   = train.drop(columns=["id"])

CATS = ["Gender", "City_Type", "Current_Car_Type",
        "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]

print(f"positive rate: {y.mean():.4f}")
train.head(3)

## 2. The leak, shown three ways

`Annual_Income_USD` is where almost all of the exploitable structure lives.

In [ ]:
inc = train["Annual_Income_USD"].to_numpy(np.int64)
km  = np.round(train["Daily_Commute_km"].to_numpy(float) * 10).astype(np.int64)
s   = pd.Series(y)

print("purchase rate by LAST DIGIT of income")
print("  (digit 0 is dragged down by the $30,000 floor spike)")
print(s.groupby(inc % 10).mean().round(4).to_string(), "\n")

mask = inc != 30_000
print("same thing with the floor removed - the spread does NOT disappear")
print(s[mask].groupby(inc[mask] % 10).mean().round(4).to_string(), "\n")

print(f"floor rows  income == 30000 : n={(~mask).sum():>7,}  rate={y[~mask].mean():.4f}")
print(f"other rows                  : n={mask.sum():>7,}  rate={y[mask].mean():.4f}")
print(f"overall                     : n={len(y):>7,}  rate={y.mean():.4f}")

In [ ]:
# How much signal does a *pure remainder* carry? Encode it out-of-fold and score it alone.
def te_auc(key, prior=100.0, folds=5):
    gm  = y.mean()
    oof = np.zeros(len(y))
    for a, b in StratifiedKFold(folds, shuffle=True, random_state=SEED).split(key, y):
        st = pd.DataFrame({"k": key[a], "y": y[a]}).groupby("k")["y"].agg(["sum", "count"])
        sm = (st["sum"] + prior * gm) / (st["count"] + prior)
        oof[b] = pd.Series(key[b]).map(sm).fillna(gm).to_numpy()
    return roc_auc_score(y, oof)

print(f"{'key':<28}{'n unique':>10}{'AUC alone':>12}")
for name, key in [("Annual_Income_USD",        inc),
                  ("income // 100",            inc // 100),
                  ("income // 1000",           inc // 1000),
                  ("income % 1000   <-- leak", inc % 1000),
                  ("income % 100",             inc % 100),
                  ("income % 10",              inc % 10),
                  ("Daily_Commute_km",         km)]:
    print(f"{name:<28}{len(np.unique(key)):>10,}{te_auc(key):>12.5f}")

`income % 1000` reaching **0.618** is the whole story. Dividing an income by 1000 and
keeping the remainder should destroy every bit of meaning it had. Here it predicts.

For contrast, the honest structural signal is concentrated in one tiny interaction:

In [ ]:
joint = (train["Environmental_Concern_Level"].astype(str) + "|" +
         train["Subsidy_Available"]).to_numpy()
print(f"Environmental_Concern_Level x Subsidy_Available")
print(f"  {len(np.unique(joint))} groups, AUC alone = {te_auc(joint, prior=50.0):.5f}")
print("\nSo the task decomposes roughly as:")
print("  0.917  <- that 10-cell interaction")
print("  0.025  <- fine-grained income + the digit leak")
print("  0.004  <- everything else, plus ensembling")

## 3. Features

Only the blocks that earned their place in the ablation:

* **digits / moduli** of income and commute distance — the leak
* **quantisation ladders** (`income // 250`, `// 500`, …) — lets a tree cut the income
  axis at several resolutions at once
* **value frequency** — how many rows share this exact number, computed over train+test.
  Unsupervised, so no label leaks; it just tells the model which values the generator
  over-produced.

Deliberately **not** included: ratios, products, `income / age`, categorical crosses,
floor indicator flags. All were measured. All were neutral or negative.

In [ ]:
def build_features(df, freq):
    out = df.copy()
    inc = out["Annual_Income_USD"].to_numpy(np.int64)
    km  = np.round(out["Daily_Commute_km"].to_numpy(float) * 10).astype(np.int64)

    # --- the leak ---
    out["inc_d1"]      = (inc % 10).astype("int8")
    out["inc_d2"]      = (inc // 10 % 10).astype("int8")
    out["inc_d3"]      = (inc // 100 % 10).astype("int8")
    out["inc_mod100"]  = (inc % 100).astype("int16")
    out["inc_mod1000"] = (inc % 1000).astype("int16")
    out["km_d1"]       = (km % 10).astype("int8")
    out["km_mod100"]   = (km % 100).astype("int8")

    # --- multi-resolution views of the same axis ---
    for d in (50, 100, 250, 500, 1000, 2500, 5000):
        out[f"inc_q{d}"] = (inc // d).astype("int32")
    for d in (5, 10, 25, 50):
        out[f"km_q{d}"] = (km // d).astype("int32")

    # --- how common is this exact value (train+test, label-free) ---
    out["fq_inc"] = pd.Series(inc).map(freq["inc"]).to_numpy("float32")
    out["fq_km"]  = pd.Series(km).map(freq["km"]).to_numpy("float32")
    return out


inc_tr = train["Annual_Income_USD"].to_numpy(np.int64)
inc_te = test["Annual_Income_USD"].to_numpy(np.int64)
km_tr  = np.round(train["Daily_Commute_km"].to_numpy(float) * 10).astype(np.int64)
km_te  = np.round(test["Daily_Commute_km"].to_numpy(float) * 10).astype(np.int64)

freq = {"inc": pd.Series(np.concatenate([inc_tr, inc_te])).value_counts(),
        "km":  pd.Series(np.concatenate([km_tr,  km_te ])).value_counts()}

X    = build_features(train, freq)
Xte  = build_features(test,  freq)
for c in CATS:
    X[c]   = X[c].astype("category")
    Xte[c] = pd.Categorical(Xte[c], categories=X[c].cat.categories)

log(f"feature matrix: {X.shape}")

## 4. Fold-safe target encoding

This is the part that is easy to get wrong, and getting it wrong inflates your CV
until the leaderboard corrects you.

The encoder is rebuilt **inside every outer fold**, fitted only on that fold's training
rows. The training rows get an inner out-of-fold encoding so a row never sees its own
label; validation and test rows get the encoding fitted on the full fold-train set.
No validation label ever reaches the encoder.

In [ ]:
PRIOR = 2.0     # swept: 200->.94515  50->.94533  10->.94542  5->.94545  2->.94547  1->.94548
TE_KEYS = {"te_inc":      lambda v: v,
           "te_inc100":   lambda v: v // 100,
           "te_inc1000":  lambda v: v // 1000,
           "te_incm1000": lambda v: v % 1000}


def _encode(k_fit, y_fit, k_apply, gm):
    st = pd.DataFrame({"k": k_fit, "y": y_fit}).groupby("k")["y"].agg(["sum", "count"])
    sm = (st["sum"] + PRIOR * gm) / (st["count"] + PRIOR)
    return pd.Series(k_apply).map(sm).fillna(gm).to_numpy("float32")


LIN_MAPS = {"Home_Charging_Possible": {"Yes": 1, "No": 0},
            "Subsidy_Available": {"Yes": 1, "No": 0},
            "Range_Anxiety_Level": {"Low": 0, "Medium": 1, "High": 2},
            "Gender": {"Male": 0, "Female": 1, "Other": 2},
            "City_Type": {"Urban": 0, "Suburban": 1, "Rural": 2},
            "Current_Car_Type": {"Sedan": 0, "SUV": 1, "Hatchback": 2, "Truck": 3}}


def linear_design(df):
    # the columns the recovered rule is linear in
    d = pd.DataFrame(index=df.index)
    d["income"] = df["Annual_Income_USD"] / 1e5
    d["concern"] = df["Environmental_Concern_Level"]
    d["commute"] = df["Daily_Commute_km"]
    d["cars"] = df["Number_of_Cars_Owned"]
    d["csh"] = df["Charging_Stations_Near_Home"]
    d["csw"] = df["Charging_Stations_Near_Work"]
    d["age"] = df["Age"]
    d["home"] = df["Home_Charging_Possible"].map(LIN_MAPS["Home_Charging_Possible"])
    d["subsidy"] = df["Subsidy_Available"].map(LIN_MAPS["Subsidy_Available"])
    anx = df["Range_Anxiety_Level"].map(LIN_MAPS["Range_Anxiety_Level"])
    d["anx_med"] = (anx == 1).astype(int)
    d["anx_high"] = (anx == 2).astype(int)
    for c in ("Gender", "City_Type", "Current_Car_Type"):
        code = df[c].map(LIN_MAPS[c])
        for v in sorted(set(LIN_MAPS[c].values()))[1:]:
            d[f"{c}_{v}"] = (code == v).astype(int)
    return d


def attach_te(A, B, C, inc_a, inc_b, inc_c, y_a):
    # A = fold-train, B = fold-valid, C = test. Encoder sees only y_a.
    gm = y_a.mean()
    inner = StratifiedKFold(5, shuffle=True, random_state=7)
    for name, fn in TE_KEYS.items():
        ka, kb, kc = fn(inc_a), fn(inc_b), fn(inc_c)
        col = np.full(len(ka), gm, dtype="float32")
        for i, j in inner.split(ka, y_a):
            col[j] = _encode(ka[i], y_a[i], ka[j], gm)
        A[name] = col
        B[name] = _encode(ka, y_a, kb, gm)
        C[name] = _encode(ka, y_a, kc, gm)
    return A, B, C

## 4b. The label rule, recovered

The original 10,000-row dataset this was generated from is itself rule-based, and
the rule is recoverable. Evidence: on the original, a plain **logistic regression
beats LightGBM** (CV AUC 0.9069 vs 0.8951). A tree only loses to a linear model
when the truth is linear.

Fitting that logistic regression and reading off the coefficients gives, with
`Environmental_Concern_Level` normalised to 1.0:

| term | recovered |
|---|---|
| `Annual_Income_USD / 1e5` | +2.17 |
| `Environmental_Concern_Level` | +1.00 |
| `Subsidy_Available == Yes` | +3.16 |
| `Range_Anxiety == Medium` | −1.48 |
| `Range_Anxiety == High` | −3.98 |

That single linear score scores **AUC 0.9375 on the synthetic training set by
itself** — no trees, no encoding, one dot product.

A boosted tree approximates a diagonal boundary with a staircase of axis-aligned
splits, so handing it the score directly ought to help. It barely does.

| setup | OOF | delta |
|---|---|---|
| 5-fold, no rule feature | 0.945466 | — |
| 5-fold, + refitted rule score | 0.945523 | +0.000057 |
| 10-fold, no rule feature | 0.945979 | — |
| **10-fold, + refitted rule score** | **0.946001** | **+0.000022** |

At 5 folds it looked like it had cleared the +0.00004 noise floor from section 8.
At 10 folds it did not. By the standard this notebook sets elsewhere, that is a
non-result, and I am keeping the feature only because it costs nothing and is the
honest version of a thing many notebooks add without measuring.

The version of the score taken from the original dataset's coefficients rather
than refitted on the training fold was worth +0.000019 — also inside the noise.

The lesson is the boring one: the trees had already found this. Knowing the exact
generative rule is worth a great deal for *understanding* the competition and
almost nothing for *scoring* in it.

## 5. Three gradient boosters

Same features, different inductive biases. Each fold reseeds so the ensemble also
averages over seed noise.

**One trap worth knowing about.** Fold predictions are pooled in *rank space* rather
than as raw probabilities. Writing `oof[valid_idx] = model.predict_proba(...)` looks
harmless, but if one fold early-stops much sooner than the others its probabilities
come out squashed into a narrow band. Every fold still scores fine on its own, and the
pooled OOF collapses anyway, because the folds are no longer on a common scale.

It happened here on a first pass — XGBoost fold 2 stopped early and returned everything
inside `[0.164, 0.219]` while the other folds spanned `[0.00003, 0.998]`:

| | per-fold AUC | pooled OOF |
|---|---|---|
| XGB, raw probabilities | 0.9433 – 0.9459 | **0.9269** |
| XGB, per-fold ranks | 0.9433 – 0.9459 | **0.9450** |

Ranking inside each fold before pooling costs nothing — AUC only cares about order —
and makes the OOF immune to it. A longer `early_stopping_rounds` (400 here) stops the
underlying problem from happening in the first place.

In [ ]:
def fit_xgb(A, ya, B, yb, C, fold):
    m = xgb.XGBClassifier(
        n_estimators=6000, learning_rate=0.015, max_depth=8,
        subsample=0.85, colsample_bytree=0.5, min_child_weight=40,
        reg_lambda=4.0, reg_alpha=0.2, max_bin=512,
        enable_categorical=True, tree_method="hist",
        device="cuda" if GPU else "cpu",
        eval_metric="auc", early_stopping_rounds=400,
        n_jobs=-1, random_state=fold)
    m.fit(A, ya, eval_set=[(B, yb)], verbose=False)
    return m.predict_proba(B)[:, 1], m.predict_proba(C)[:, 1]


def fit_lgb(A, ya, B, yb, C, fold):
    m = lgb.LGBMClassifier(
        n_estimators=8000, learning_rate=0.015, num_leaves=160,
        min_child_samples=80, subsample=0.85, subsample_freq=1,
        colsample_bytree=0.5, reg_lambda=4.0, max_bin=511,
        n_jobs=-1, random_state=fold, verbose=-1)
    m.fit(A, ya, eval_set=[(B, yb)], eval_metric="auc",
          callbacks=[lgb.early_stopping(400, verbose=False)])
    return m.predict_proba(B)[:, 1], m.predict_proba(C)[:, 1]


def fit_cat(A, ya, B, yb, C, fold):
    m = CatBoostClassifier(
        iterations=8000, learning_rate=0.04, depth=8, l2_leaf_reg=6,
        eval_metric="AUC", random_seed=fold, od_type="Iter", od_wait=400,
        task_type="GPU" if GPU else "CPU",
        verbose=0, allow_writing_files=False)
    m.fit(Pool(A, ya, cat_features=CATS),
          eval_set=Pool(B, yb, cat_features=CATS), use_best_model=True)
    return (m.predict_proba(B)[:, 1],
            m.predict_proba(Pool(C, cat_features=CATS))[:, 1])


FITTERS = {"xgb": fit_xgb, "lgb": fit_lgb, "cat": fit_cat}

In [ ]:
L    = linear_design(train)          # defined in section 4b, above
Lte  = linear_design(test)
log(f"linear design: {L.shape}")

rk   = lambda v: rankdata(v) / len(v)
oof  = {k: np.zeros(len(X))   for k in FITTERS}
pte  = {k: np.zeros(len(Xte)) for k in FITTERS}
cv   = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)

for fold, (i_tr, i_va) in enumerate(cv.split(X, y)):
    A, B, C = X.iloc[i_tr].copy(), X.iloc[i_va].copy(), Xte.copy()
    A, B, C = attach_te(A, B, C, inc_tr[i_tr], inc_tr[i_va], inc_te, y[i_tr])
    ya, yb = y[i_tr], y[i_va]

    # the recovered linear rule, refitted on this fold's training rows only
    lr = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))
    lr.fit(L.iloc[i_tr], ya)
    A["rule_score"] = lr.decision_function(L.iloc[i_tr]).astype("float32")
    B["rule_score"] = lr.decision_function(L.iloc[i_va]).astype("float32")
    C["rule_score"] = lr.decision_function(Lte).astype("float32")

    for name, fn in FITTERS.items():
        o, p = fn(A, ya, B, yb, C, fold)
        oof[name][i_va] = rk(o)            # rank inside the fold, then pool
        pte[name] += rk(p) / N_SPLITS

    log("fold %d  " % fold + "  ".join(
        f"{k}={roc_auc_score(yb, oof[k][i_va]):.5f}" for k in FITTERS))
    del A, B, C; gc.collect()

print()
for k in FITTERS:
    print(f"{k.upper():<4} OOF AUC = {roc_auc_score(y, oof[k]):.6f}")

## 6. Blending

Rank-average, with weights chosen by greedy hill climbing on the OOF predictions.
Ranks rather than probabilities because AUC only cares about ordering, and the three
models are calibrated differently.

In [ ]:
def hill_climb(sources, y_true, steps=60):
    # Greedy forward selection with replacement; returns normalised weights.
    keys = list(sources)
    R = {k: rankdata(sources[k]) / len(y_true) for k in keys}
    w, cur, best = {k: 0 for k in keys}, np.zeros(len(y_true)), 0.0
    for s in range(steps):
        auc, pick = max((roc_auc_score(y_true, cur * s / (s + 1) + R[k] / (s + 1)), k)
                        for k in keys)
        if s and auc <= best:
            break
        cur, best = cur * s / (s + 1) + R[pick] / (s + 1), auc
        w[pick] += 1
    tot = sum(w.values()) or 1
    return {k: v / tot for k, v in w.items()}, best


W, blend_oof = hill_climb(oof, y)
print("weights:", {k: round(v, 3) for k, v in W.items() if v})
print(f"BLEND OOF AUC = {blend_oof:.6f}")

n_te = len(Xte)
own_test = sum(W[k] * rankdata(pte[k]) / n_te for k in pte)
own_oof  = sum(W[k] * rankdata(oof[k]) / len(y) for k in oof)

## 7. Optional: fold in the public ensembles

The strongest public submissions correlate **0.999** with each other, so mixing them
together buys almost nothing. This pipeline is the diverse ingredient — it is built on
a different feature set entirely — which is exactly what a blend is short of.

If you attach any public submissions as notebook inputs, the cell below picks them up
and rank-averages them in. If you attach nothing, it is a no-op and the notebook still
produces a complete from-scratch submission.

`PUBLIC_WEIGHT` is the share given to the public pool. Skip this section entirely by
setting it to `0.0`.

In [ ]:
ANCHOR_WEIGHT = 0.80   # share for the single strongest public submission
POOL_WEIGHT   = 0.20   # shared by everything else, this notebook included
                       # set both to 0 for a pure from-scratch submission

# The strongest public artefact at time of writing is jazivxt's zoom-zoom
# baseline (public LB 0.94635). Anything else that looks like a submission and
# lines up with the test ids joins the diversity pool.
ANCHOR_HINTS = ("curvature", "zoom")

anchor, pool, seen = None, [], []
for path in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)):
    base = os.path.basename(path).lower()
    if "sample" in base or ("submission" not in base and "curvature" not in base):
        continue
    try:
        d = pd.read_csv(path)
    except Exception:
        continue
    if len(d) != len(test_id) or not {"id", TARGET} <= set(d.columns):
        continue
    v = d.sort_values("id")[TARGET].to_numpy()
    if any(np.array_equal(v, s) for s in seen):      # same file under two names
        continue
    seen.append(v)
    r = rankdata(v) / len(v)
    if anchor is None and any(h in path.lower() for h in ANCHOR_HINTS):
        anchor = r
        print(f"anchor : {path}")
    else:
        pool.append(r)
        print(f"pool   : {path}")

pool.append(rankdata(own_test) / n_te)               # this notebook joins the pool
print(f"\nanchor found: {anchor is not None} | pool members: {len(pool)}")

if anchor is None and pool:
    anchor, pool = pool[0], pool[1:]                 # no known anchor: promote one

if anchor is not None and (ANCHOR_WEIGHT + POOL_WEIGHT) > 0:
    pooled = rankdata(np.mean(pool, axis=0)) / n_te if pool else anchor
    final = rankdata(ANCHOR_WEIGHT * anchor + POOL_WEIGHT * pooled) / n_te
    print(f"blended: {ANCHOR_WEIGHT:.2f} anchor + {POOL_WEIGHT:.2f} pool of {len(pool)}")
else:
    final = own_test
    print("using the from-scratch prediction alone")

Weights are not tuned, on purpose. Section 8 shows that differences below
**+0.00004** are not resolvable on this leaderboard, and a sweep over the anchor
weight moved OOF by +0.000026 while moving the public LB by −0.00003. Anything
in the 0.7–0.9 range lands in the same place; picking the argmax of a noisy
curve is how you overfit a leaderboard.

### Credit

The blended score is not mine. Six other people's public work is doing most of
the lifting, and the pipeline above contributes one diverse member and the
weighting. Attach these as inputs (**Add Input → Datasets / Notebook Output**)
to reproduce it, and please upvote the originals:

| source | author | role |
|---|---|---|
| [s6e9-zoom-zoom-baseline](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline) | [@jazivxt](https://www.kaggle.com/jazivxt) | the anchor, public LB 0.94635 |
| [PS-s6e9 h-blend](https://www.kaggle.com/code/nina2025/ps-s6e9-h-blend-1) | [@nina2025](https://www.kaggle.com/nina2025) | pool |
| [S6E9 OOF CV 0.94618](https://www.kaggle.com/code/najiama/s6e9-electric-vehicle-oof-cv-0-94618-lb-0-94633) | [@najiama](https://www.kaggle.com/najiama) | pool, and the OOF files that made honest weighting possible |
| [Single XGB CV 0.94583](https://www.kaggle.com/code/evgendvorkin/s6e9-single-xgb-cv-0-94583) | [@evgendvorkin](https://www.kaggle.com/evgendvorkin) | pool; the digit-feature idea started here |
| [Transformer + GBDT](https://www.kaggle.com/code/lamhuy8904/s6e9-94-6-transformer-and-gbdt-ensemble) | [@lamhuy8904](https://www.kaggle.com/lamhuy8904) | pool |
| [Stacked LGBM/CatB/XGB/HGBC](https://www.kaggle.com/code/kospintr/evehicle-stacked-lgbm-catb-xgb-hgbc-baseline) | [@kospintr](https://www.kaggle.com/kospintr) | pool |
| [S6E9 LightGBM](https://www.kaggle.com/code/kirill0212/s6e9-lightgbm) | [@kirill0212](https://www.kaggle.com/kirill0212) | pool |

Also [@yhay81](https://www.kaggle.com/code/yhay81/exact-reconstruction-of-the-ev-source-dataset),
whose exact reconstruction of the source generator is what lets this notebook say
*why* the digit features work instead of just that they do.

**With nothing attached the notebook still runs end to end** and submits its own
model, which measured **0.94592** on the public LB. That number is the honest one
for the code in this notebook; **0.94635** is what it becomes once six other
people's submissions are averaged in.

In [ ]:
sub = pd.DataFrame({"id": test_id, TARGET: final})
sub.to_csv("submission.csv", index=False)
log(f"wrote submission.csv {sub.shape}")
sub.head()

## 8. How small a difference can this leaderboard actually see?

Worth knowing before you spend a week on the fourth decimal place.

Take two candidate rankings whose full-OOF AUCs differ by a known amount,
resample leaderboard-sized subsets, and look at how the measured difference moves:

| LB split | rows | mean diff | sd | P(diff > 0) |
|---|---|---|---|---|
| 10% | 28,657 | +0.000025 | 0.000026 | 82.5% |
| 20% | 57,314 | +0.000026 | 0.000020 | 92.0% |
| 50% | 143,285 | +0.000025 | 0.000011 | 98.5% |
| 100% | 286,571 | +0.000025 | 0.000007 | 100% |

On a 20% public split the sampling sd of a *difference* is 0.000020, so you need
roughly **+0.00004** before a gain is distinguishable from luck — and that is the
bar for it carrying to the private split, which is different rows.

Two things I measured that sit under that bar:

* a blend reweighting worth **+0.000022** OOF moved the public LB by **−0.00003**
* a second one worth **+0.000026** OOF moved it by **+0.00001**

Both are noise. If your CV improves by 0.00002, you have not improved anything.

**And the ceiling is close.** Ranking by an estimate of the true `P(y=1|x)` tops out
around 0.946–0.947 on this data — the label is close to a coin flip given `x`, which
is why the entire leaderboard is stacked inside 0.0005 of itself. Whatever is left to
win here is smaller than most people's error bars.

A neural net was the obvious thing to add for decorrelation. It measured worse:
MLP with categorical embeddings reached OOF 0.9446, and adding it to the GBDT pool
moved the pool from 0.945973 to 0.945891, i.e. **−0.00008**. Reported because it
failed; the diversity that helps here has to come from a model that is actually
competitive, not merely different.

## What to take away

1. **`Annual_Income_USD % 1000` predicts the target.** In real survey data that is
   impossible. Check the low-order digits of every continuous column in a synthetic
   Playground dataset before you tune anything.
2. **Target-encode income at several resolutions** — raw, `//100`, `//1000`, `%1000`.
   Worth about as much as the digit columns themselves, and the two stack.
3. **Turn the smoothing down.** Prior 50 → 0.945327, prior 2 → 0.945466. The leak lives
   at fine value resolution; heavy smoothing averages it away.
4. **Refit the encoder inside every fold.** Encoding once on the full training set before
   cross-validating leaks validation labels into training rows.
5. **Ratios and interaction products cost you score here.** They were measured, not assumed.
6. **Pool fold predictions in rank space**, or one early-stopped fold will quietly
   destroy your OOF.
7. **Know your noise floor before you optimise.** Below ~0.00004, this leaderboard
   cannot tell you apart from yourself.

Things that were tried and did **not** work, so you can skip them: neighbourhood /
curvature statistics on binned income (+0.000016 on top of the target encoding),
extra target-encoding keys (all negative), categorical crosses (−0.00005), floor
indicator flags (+0.00001), and an embedding MLP for ensemble diversity (−0.00008
to the pool).

If this was useful, an upvote helps. Forks and disagreements welcome — the ablation
table is the interesting part, and I would rather it be checked than believed.